#**Model Architecture: SimCLR (Simple Framework for Contrastive Learning)**
This implementation uses SimCLR, a self-supervised learning approach that learns useful visual representations by maximizing the agreement between differently augmented views of the same data example.


###**Data Loading Strategy**
To facilitate the self-supervised training objective, a custom dataset class, SimCLRDataset, was implemented.

In [ ]:
class SimCLRDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # 1. Load the original image
        img_path = self.dataframe.iloc[idx]['path_img'] # Make sure the column is correct
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


        # 2. Apply the transformation TWICE independently
        # This creates two different "views" of the same concept
        if self.transform:
            view1 = self.transform(image=image)['image']
            view2 = self.transform(image=image)['image']
        else:
            # Fallback if there are no transformations (should not happen in SimCLR)
            view1 = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            view2 = view1.clone()


        return view1, view2

###**Stochastic Augmentation Pipeline**
A robust data augmentation strategy is crucial for Contrastive Learning to prevent the model from learning trivial features (shortcuts) such as color histograms or edge artifacts. We implemented a "strong" augmentation pipeline using the Albumentations library.

In [ ]:
# --- STRONG AUGMENTATIONS (Crucial for SimCLR) ---
# SimCLR only works if the two views are different enough
# to force the model to "understand" the content.

simclr_transform = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.5, 1.0)), # Aggressive zoom
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=90, p=0.5),
    # Color Jitter: Simulates staining color variations
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
    A.ToGray(p=0.2),
    A.GaussianBlur(blur_limit=(3, 7), p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

###**Network Architecture (SimCLR Module)**
The model architecture is defined in the SimCLR class, which adapts a standard Convolutional Neural Network (CNN) for the contrastive learning task. The architecture is composed of two primary stages: the Backbone Encoder and the Projection Head.

What we are using here:


*   **Backbone Encoder**: We utilize ResNet-50 as the base encoder to extract high-level representations from images. The backbone is initialized with weights pre-trained on ImageNet (IMAGENET1K_V1). Although SimCLR is a self-supervised method, initializing with pre-trained weights can accelerate convergence and provide a robust starting point for feature extraction. The original fully connected classification layer (fc) of the ResNet-50 is replaced with an Identity layer. This allows us to access the raw feature vector directly from the penultimate layer, which has a dimensionality of d=2048.
*   **Projection Head**: Following the encoder, a non-linear projection head maps the representation h into a lower-dimensional latent space z where the contrastive loss is calculated.



In [ ]:
class SimCLR(nn.Module):
    def __init__(self, base_model, projection_dim=128):
        super(SimCLR, self).__init__()


        # 1. Load the backbone (e.g. ResNet50)
        # Note: We use ImageNet weights as a good starting point
        self.backbone = base_model(weights=models.ResNet50_Weights.IMAGENET1K_V1)


        # 2. Replace the last layer (fc) with Identity (to extract raw features)
        # ResNet50 has 2048 output features before the FC
        self.n_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()


        # 3. Projection Head (MLP: Linear -> ReLU -> Linear)
        # Projects features into a space where we compute similarity
        self.projection_head = nn.Sequential(
            nn.Linear(self.n_features, 512),
            nn.ReLU(),
            nn.Linear(512, projection_dim)
        )


    def forward(self, x):
        h = self.backbone(x)       # Representation (Features) - used later
        z = self.projection_head(h) # Projections - used for Loss now
        return h, z


**Loss Function: NT-Xent** (Normalized Temperature-scaled Cross Entropy)
The training objective is defined by the NTXentLoss class, which implements the contrastive loss function used to optimize the SimCLR model.

The loss function takes the projection vectors $z_i$ and $z_j$ (corresponding to the two augmented views of the batch) as input and performs the following operations:Similarity Matrix Construction: It concatenates the two sets of vectors into a single tensor of size $2N$ and computes the pairwise cosine similarity between all samples. The result is scaled by a temperature parameter ($\tau$, set here to 0.5) to control the sharpness of the distribution.

The problem is formulated as a classification task using standard CrossEntropyLoss. The model tries to classify the correct positive pair out of all possible pairs in the batch. By minimizing this loss, the model maximizes the similarity between positive pairs while pushing away negative examples in the latent space.

In [ ]:
class NTXentLoss(nn.Module):
    def __init__(self, batch_size, temperature=0.5, device='cuda'):
        super().__init__()
        self.batch_size = batch_size
        self.temperature = temperature
        self.device = device
        self.criterion = nn.CrossEntropyLoss(reduction="sum")
        self.similarity_f = nn.CosineSimilarity(dim=2)


    def mask_correlated_samples(self, batch_size):
        # Create a mask to ignore similarity of an image with itself
        N = 2 * batch_size
        mask = torch.ones((N, N), dtype=bool)
        mask = mask.fill_diagonal_(0)
        for i in range(batch_size):
            mask[i, batch_size + i] = 0
            mask[batch_size + i, i] = 0
        return mask


    def forward(self, z_i, z_j):
        """
        z_i, z_j: Output of the projection head for the two views. Shape [Batch, Dim]
        """
        N = 2 * self.batch_size
        z = torch.cat((z_i, z_j), dim=0)

        # Cosine similarity matrix
        sim = self.similarity_f(z.unsqueeze(1), z.unsqueeze(0)) / self.temperature

        # Labels: for each image i, the positive target is (i + batch_size)
        sim_i_j = torch.diag(sim, self.batch_size)
        sim_j_i = torch.diag(sim, -self.batch_size)

        positive_samples = torch.cat((sim_i_j, sim_j_i), dim=0).reshape(N, 1)
        negative_samples = sim[self.mask_correlated_samples(self.batch_size)].reshape(N, -1)

        labels = torch.zeros(N).to(self.device).long()
        logits = torch.cat((positive_samples, negative_samples), dim=1)

        loss = self.criterion(logits, labels)
        return loss / N


#####**Generation test patches logic (skip)**

Here there should be the logic to generate the patches for the test set, this is removed for brevity. Basically we used both pacthes from training and test sets since this is a self-supervised learning practice and avoids every type of data leakage.


###**Configuration - Contrastive Pretraining**

In [ ]:
# --- CONFIGURATION ---
BATCH_SIZE = 64 # SimCLR likes large batches.
LR = 1e-4 # Usually a lower learning rate
EPOCHS = 10 # Even a few epochs help a lot
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset & Loader
# You can use the WHOLE dataset (Train + Test if you want) because it is (self)unsupervised!
full_df = pd.concat([train_df, test_df])
simclr_ds = SimCLRDataset(full_df, transform=simclr_transform)
simclr_loader = DataLoader(simclr_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2)

# Model & Loss
base_resnet = models.resnet50 # Or your preferred architecture
model = SimCLR(base_resnet).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = NTXentLoss(batch_size=BATCH_SIZE, device=DEVICE)

print("Starting Contrastive Pretraining...")

for epoch in range(EPOCHS):
    total_loss = 0
    model.train()

    for view1, view2 in tqdm(simclr_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        view1 = view1.to(DEVICE)
        view2 = view2.to(DEVICE)

        optimizer.zero_grad()

        # Forward pass
        _, z1 = model(view1) # We are interested only in the projections z here
        _, z2 = model(view2)

        # Calculate Loss
        loss = loss_fn(z1, z2)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss / len(simclr_loader):.4f}")

# --- SAVING THE "ENLIGHTENED" WEIGHTS ---
# Save only the backbone, discard the projection head
torch.save(model.backbone.state_dict(), "resnet50_simclr_pretrained.pth")
print("Pretrained backbone saved!")


🚀 Starting Contrastive Pretraining...


Epoch 1/10: 100%|██████████| 200/200 [04:24<00:00,  1.32s/it]


Epoch 1 Loss: 3.4373


Epoch 2/10: 100%|██████████| 200/200 [04:09<00:00,  1.25s/it]


Epoch 2 Loss: 3.2351


...

Epoch 8 Loss: 3.1326


Epoch 9/10: 100%|██████████| 200/200 [04:10<00:00,  1.25s/it]


Epoch 9 Loss: 3.1264


Epoch 10/10: 100%|██████████| 200/200 [04:09<00:00,  1.25s/it]


Epoch 10 Loss: 3.1243
✅ Backbone pre-addestrata salvata!


Here we saved our pre-trained backbone *"resnet50_simclr_pretrained.pth"* that was used in the following cell.

##**Application for the model initialization and training**

In [ ]:
# DATA LOADING AND WEIGHT CALCULATION
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Make sure the dataframe is in memory
if 'df_patches' not in locals():
    try:
        df_patches = pd.read_csv('dataset_patches_5classes.csv')
        print("✅ DataFrame loaded from CSV.")
    except FileNotFoundError:
        print("⚠️ ERROR: 'dataset_patches_5classes.csv' not found. Have you done the patching?")


# Map of the 5 classes
class_map = {
    'Triple negative': 0,
    'Luminal A': 1,
    'Luminal B': 2,
    'HER2(+)': 3,
    'Healthy': 4
}


print("Calculating weights for the 5 classes...")
# Convert labels to numbers
if 'df_patches' in locals():
    y_train_indices = df_patches['label'].map(class_map).values
    classes_unique = np.unique(y_train_indices)
    weights_np = compute_class_weight(class_weight='balanced', classes=classes_unique, y=y_train_indices)
    weights_tensor = torch.tensor(weights_np, dtype=torch.float32).to(DEVICE)
    print(f"Weights calculated: {weights_np}")
else:
    print("⚠️ df_patches not available, skipping weight calculation.")


# --- 2. TRAINING CONFIGURATION ---
NUM_EPOCHS = 50
PATIENCE = 10
WARMUP_EPOCHS = 5  # <--- NEW: Number of epochs with backbone frozen
SAVE_PATH = '/content/drive/MyDrive/2nd Challenge/dual_stream_5classes_CIA_3.pth'


# --- 3. MODEL (5 CLASSES) ---
class DualStreamResNet(nn.Module):
    def __init__(self, num_classes=5):
        super(DualStreamResNet, self).__init__()

        weights_50 = models.ResNet50_Weights.DEFAULT
        self.rgb_stream = models.resnet50(weights=weights_50)
        self.rgb_stream.fc = nn.Identity()

        weights_18 = models.ResNet18_Weights.DEFAULT
        self.mask_stream = models.resnet18(weights=weights_18)
        self.mask_stream.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        with torch.no_grad():
            self.mask_stream.conv1.weight[:] = torch.mean(
                models.resnet18(weights=weights_18).conv1.weight, dim=1, keepdim=True
            )

        self.mask_stream.fc = nn.Identity()

        # Here we used a different dropout value
        dropout_rate = 0.65

        self.classifier = nn.Sequential(
            # This Dropout acts on the combined features (2560) BEFORE entering the first dense layer
            nn.Dropout(p=dropout_rate),

            nn.Linear(2560, 512),
            nn.ReLU(),

            # This Dropout acts on intermediate features (512) BEFORE the final classification
            nn.Dropout(p=dropout_rate),

            nn.Linear(512, num_classes)
        )


    def forward(self, img, mask):
        f_rgb = self.rgb_stream(img)
        f_mask = self.mask_stream(mask)
        combined = torch.cat((f_rgb, f_mask), dim=1)
        return self.classifier(combined)


# B. WEIGHTS LOADING FUNCTION
def load_simclr_into_rgb_stream(target_model, path_to_weights, device):
    print(f"🔄 Attempting to load SimCLR weights from: {path_to_weights}")
    if not os.path.exists(path_to_weights):
        print(f"❌ FILE NOT FOUND: {path_to_weights}")
        return target_model

    checkpoint = torch.load(path_to_weights, map_location=device)
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key
        if new_key.startswith('module.'): new_key = new_key[7:]
        if new_key.startswith('backbone.'): new_key = new_key[9:]
        if "projection_head" in key or "fc" in key: continue
        new_state_dict[new_key] = value


    msg = target_model.rgb_stream.load_state_dict(new_state_dict, strict=False)
    print(f"✅ SUCCESS: SimCLR weights loaded into RGB branch! ({len(new_state_dict)} keys)")
    return target_model


# C. INITIALIZATION
print("Building Hybrid Model...")
model = DualStreamResNet(num_classes=5).to(DEVICE)


SIMCLR_PATH = '/content/drive/MyDrive/2nd Challenge/resnet50_simclr_pretrained.pth'
model = load_simclr_into_rgb_stream(model, SIMCLR_PATH, DEVICE)


# --- INITIAL FREEZING (WARM-UP) ---
print("🔒 Freezing Backbones for the first epochs (Warm-up)...")
for param in model.rgb_stream.parameters():
    param.requires_grad = False
for param in model.mask_stream.parameters():
    param.requires_grad = False
print("    -> Only the classifier will be trained initially.")


# --- 4. OPTIMIZER & LOSS ---
# Note: The optimizer will see requires_grad=False and will not update frozen weights
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)


if 'weights_tensor' not in globals():
    print("⚠️ weights_tensor not found! Using normal unweighted Loss.")
    criterion = nn.CrossEntropyLoss()
else:
    criterion = nn.CrossEntropyLoss(weight=weights_tensor.to(DEVICE))
    print("✅ Loss Function configured with BALANCED WEIGHTS.")


print("\n--- Model Ready for Training (Section 5) ---")


best_val_f1 = float('-inf')
early_stopping_counter = 0


for epoch in range(NUM_EPOCHS):
    start_time = time.time()


    # ---  BACKBONE UNFREEZE LOGIC  ---
    if epoch == WARMUP_EPOCHS:
        print(f"\n🔔 EPOCH {epoch+1}: FULL UNFREEZE!")
        print("    -> The backbones are now trained.")
        for param in model.rgb_stream.parameters():
            param.requires_grad = True
        for param in model.mask_stream.parameters():
            param.requires_grad = True


    # === TRAIN ===
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    train_preds = []
    train_targets = []


    # Assume train_aug_loader is defined in the global context
    for images, masks, labels, _ in train_aug_loader:
        images, masks, labels = images.to(DEVICE), masks.to(DEVICE), labels.to(DEVICE)


        optimizer.zero_grad()
        outputs = model(images, masks)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        train_preds.extend(predicted.cpu().numpy())
        train_targets.extend(labels.cpu().numpy())


    avg_train_loss = train_loss / len(train_aug_loader)
    acc_train = 100 * correct_train / total_train
    f1_train = f1_score(train_targets, train_preds, average='macro', zero_division=0) * 100


    # === VALIDATION ===
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    val_preds = []
    val_targets = []


    with torch.no_grad():
        for images, masks, labels, _ in val_aug_loader:
            images, masks, labels = images.to(DEVICE), masks.to(DEVICE), labels.to(DEVICE)
            outputs = model(images, masks)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            val_preds.extend(predicted.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())


    avg_val_loss = val_loss / len(val_aug_loader)
    acc_val = 100 * correct_val / total_val
    f1_val = f1_score(val_targets, val_preds, average='macro', zero_division=0) * 100


    scheduler.step()
    elapsed = time.time() - start_time


    # Visual indicator if we are in Warm-up or Fine-tuning
    mode_str = "WARM-UP" if epoch < WARMUP_EPOCHS else "FINE-TUNE"


    print(f"Epoch {epoch+1}/{NUM_EPOCHS} [{mode_str}] [{elapsed:.0f}s] | "
          f"TRAIN: Acc={acc_train:.1f}% F1={f1_train:.1f}% | "
          f"VAL: Acc={acc_val:.1f}% F1={f1_val:.1f}% | "
          f"Loss: {avg_train_loss:.3f}/{avg_val_loss:.3f}", end="")


    if f1_val > best_val_f1:
        best_val_f1 = f1_val
        early_stopping_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(" --> SAVED (Best F1)")
    else:
        early_stopping_counter += 1
        print(f" | No Improvement ({early_stopping_counter}/{PATIENCE})")


        if early_stopping_counter >= PATIENCE:
            print(f"\nSTOP: Early Stopping activated.")
            break


print("\nEnd of Training.")
